# 第 5 周 · 周四：JOIN 全家桶——不只是 LEFT JOIN

> 难度：⭐⭐⭐ | 今日目标：掌握各种JOIN的适用场景，避免最常见的JOIN陷阱
> 配对课程：01python_learn W5D4（文本处理进阶）
> 前置依赖：基础SELECT/GROUP BY | 后续用途：留存SQL、数仓星型模型多表JOIN
> 扩展阅读：02analyst_thinking/week_01/day_03（JOIN全家桶与陷阱完整知识图谱）
> 面试地位：大厂SQL面试必考，约占SQL考题的25%

---

## 一、核心概念与第一性原理

### 1.1 JOIN 的本质——笛卡尔积加过滤条件

JOIN的本质是集合论中的**笛卡尔积加上过滤条件**。理解这一点是掌握JOIN的钥匙。

笛卡尔积是两个集合的所有可能组合。假设表A有2行（id:1,2），表B有3行（id:1,2,3），笛卡尔积产生2×3=6行。JOIN就是在这6行的基础上，按ON条件过滤出满足条件的行。

```
A × B = {(a1,b1), (a1,b2), (a1,b3), (a2,b1), (a2,b2), (a2,b3)}
然后 ON A.id = B.id → 只保留 a1,b1 和 a2,b2
```

### 1.2 四种JOIN类型——韦恩图记忆法

```
INNER JOIN：两边都有才保留          ← 最安全，只取交集
LEFT JOIN：左边全保留，右边没有填NULL  ← 最常用，找"有但没有"的模式
RIGHT JOIN：右边全保留               ← 等于LEFT JOIN换顺序
FULL OUTER JOIN：两边都全保留       ← 找"差异"时用
```

| JOIN类型 | 数学含义 | 结果行数（A=5行，B=4行，2行能匹配） |
|---------|---------|----------------------------------|
| INNER JOIN | 交集 | 2行（只有匹配的） |
| LEFT JOIN | A∪(A∩B) | 5行（A全部保留） |
| RIGHT JOIN | B∪(A∩B) | 4行（B全部保留） |
| FULL JOIN | 并集 | 7行（5+4-2=7） |

### 1.3 最致命的陷阱——LEFT JOIN + WHERE右表条件 = INNER JOIN

这是SQL中最常见的错误，也是面试最高频的考点。

**原因在于SQL的执行顺序**：
1. ON条件在JOIN过程中生效——决定哪些行能匹配上
2. WHERE条件在JOIN之后生效——过滤整个结果集

```sql
-- 写法A：过滤条件在ON里 ✓
SELECT * 
FROM orders o
LEFT JOIN users u ON o.user_id = u.user_id AND u.status = 'active'
-- 结果：所有订单都在，非活跃用户的用户信息为NULL

-- 写法B：过滤条件在WHERE里 ✗（陷阱！）
SELECT * 
FROM orders o
LEFT JOIN users u ON o.user_id = u.user_id
WHERE u.status = 'active'
-- 结果：只有活跃用户的订单！WHERE把NULL也过滤掉了
-- 这个LEFT JOIN等价于INNER JOIN！
```

**记忆口诀**：右表的过滤条件放ON，左表的过滤条件放WHERE。

### 1.4 陷阱2：一对多JOIN导致行数膨胀（最隐蔽的bug）

用户表100万行，每个用户平均10个收货地址：
```sql
SELECT * FROM users LEFT JOIN addresses ON users.user_id = addresses.user_id
```
结果不是100万行——而是接近1000万行！

**如果你接下来做SUM(amount)**：这个用户的amount会被重复加10次！

**解决方法**：先聚合再JOIN
```sql
SELECT u.*, COALESCE(a.addr_count, 0) AS addr_count
FROM users u
LEFT JOIN (
    SELECT user_id, COUNT(*) AS addr_count
    FROM addresses
    GROUP BY user_id
) a ON u.user_id = a.user_id;
```

### 1.5 陷阱3：多表JOIN顺序不同结果不同

```sql
-- A LEFT JOIN B LEFT JOIN C
-- = (A LEFT JOIN B) LEFT JOIN C
-- 如果B中有NULL，后续LEFT JOIN C时也会保留这些NULL行
```

每次LEFT JOIN都可能保留上一轮匹配不上的行。多表JOIN时，先想清楚每次JOIN的预期。

### 1.6 SQL的逻辑执行顺序与JOIN的关系

| 步骤 | 子句 | 作用 |
|------|------|------|
| 1 | FROM + JOIN | 笛卡尔积 + ON过滤 |
| 2 | WHERE | 过滤JOIN后的结果 |
| 3 | GROUP BY | 分组 |
| 4 | HAVING | 过滤组 |
| 5 | SELECT | 选择列 |
| 6 | ORDER BY | 排序 |

**ON在步骤1中生效，WHERE在步骤2中生效——这就是陷阱的根源。**

### 1.7 面试高频考点

**面试官常问**：
- "INNER JOIN和LEFT JOIN的区别？"
- "为什么LEFT JOIN + WHERE右表条件会变成INNER JOIN？"
- "NOT IN和NOT EXISTS有什么区别？"
- "一对多JOIN会导致什么问题？如何解决？"

**回答要点**：
- INNER JOIN只保留交集，LEFT JOIN保留左表全部
- WHERE在JOIN后执行，会过滤NULL行
- NOT IN有NULL陷阱（子查询有NULL返回空），NOT EXISTS没有
- 一对多JOIN导致行数膨胀，需先聚合再JOIN

> **今天最重要的认知**：做JOIN之前，问自己两句话——"输出的一行代表什么？JOIN之后行数会不会膨胀？"这两个问题能帮你避免90%的JOIN事故。

---

## 二、知识点详细拆解

### 2.1 ON vs WHERE 的执行时机——完整理解

**为什么重要**：这是理解LEFT JOIN陷阱的关键，也是面试最高频题目。

| 条件位置 | 执行时机 | 对LEFT JOIN的影响 |
|---------|---------|------------------|
| ON | JOIN时执行 | 决定哪些行匹配，不匹配的右表列填NULL |
| WHERE | JOIN后执行 | 过滤所有行，包括NULL行 |

**实战决策树**：
- 如果条件是"这个关联是否存在"→ ON
- 如果条件是"结果集应该有哪些行"→ WHERE
- 右表的业务过滤条件→ ON（保护LEFT JOIN语义）
- 左表的过滤条件→ WHERE

### 2.2 一对多JOIN的行数膨胀——全面分析

**为什么重要**：行数膨胀不会报错，只会静默地让你的统计值偏大N倍。

**判断JOIN后行数**：
- 左表1行匹配右表N行 → JOIN后N行（左表数据被复制N份）
- 如果左表某行在右表没匹配 → LEFT JOIN保留1行，INNER JOIN丢弃

**典型场景**：
- 用户表 JOIN 订单表 → 按用户统计聚合（一个用户多行订单）
- 订单表 JOIN 退款表 → 一个订单可能有多次退款
- 用户表 JOIN 收货地址表 → 一个用户可能有多个地址

**处理策略**：
| 策略 | 适用场景 |
|------|---------|
| 先聚合再JOIN | 不需要右表的明细数据 |
| COUNT(DISTINCT) | 需要明细但统计时避免重复 |
| 确认膨胀是否合理 | 需要保留明细供下游使用 |

### 2.3 多表JOIN的过滤条件位置

**为什么重要**：条件放错位置会导致错误结果。

```sql
-- 正确：各表的业务过滤放在各自的ON中
SELECT *
FROM orders o
LEFT JOIN users u ON o.user_id = u.user_id AND u.city = '北京'
LEFT JOIN products p ON o.product_id = p.product_id AND p.category = '数码'
WHERE o.order_date >= '2026-01-01';  -- 左表的过滤放WHERE
```

### 2.4 自连接（Self Join）——同一表内比较

**为什么重要**：自连接是处理层级数据和行间比较的必备技巧。

**例子1：找出工资比经理高的员工**
```sql
SELECT e.name AS employee, e.salary AS emp_salary,
       m.name AS manager, m.salary AS mgr_salary
FROM employees e
JOIN employees m ON e.manager_id = m.emp_id
WHERE e.salary > m.salary;
```

**例子2：找出连续两天登录的用户**
```sql
SELECT DISTINCT a.user_id
FROM logins a
JOIN logins b ON a.user_id = b.user_id
              AND DATE(b.login_date, '-1 day') = a.login_date;
```

### 2.5 CROSS JOIN——笛卡尔积

**为什么重要**：笛卡尔积在某些场景下有用（如生成日期×城市组合），但要小心使用。

```sql
-- 生成所有日期和品类的组合（用于补全缺失数据）
SELECT d.date, c.category
FROM dates d
CROSS JOIN categories c;
-- 行数 = dates行数 × categories行数
```

### 2.6 SEMI JOIN / ANTI JOIN —— NOT IN的NULL陷阱

**为什么重要**：NOT IN是SQL中最隐蔽的bug之一。

```sql
-- 假设refunds表中有order_id=NULL

-- NOT IN陷阱：返回空结果！
SELECT * FROM orders
WHERE order_id NOT IN (SELECT order_id FROM refunds);
-- 原因：NOT IN需要所有值都不等于，而 NULL != 1 返回 UNKNOWN
-- 只要子查询有NULL，NOT IN结果就是空集

-- 安全写法1：NOT EXISTS
SELECT * FROM orders o
WHERE NOT EXISTS (
    SELECT 1 FROM refunds r WHERE r.order_id = o.order_id
);

-- 安全写法2：LEFT JOIN + IS NULL
SELECT o.*
FROM orders o
LEFT JOIN refunds r ON o.order_id = r.order_id
WHERE r.order_id IS NULL;
```

**NOT IN vs NOT EXISTS vs LEFT JOIN IS NULL**：
| 方法 | NULL安全 | 性能 | 推荐 |
|------|---------|------|------|
| NOT IN | ❌ | 一般 | 不推荐 |
| NOT EXISTS | ✅ | 好 | 推荐 |
| LEFT JOIN + IS NULL | ✅ | 好 | 推荐 |

### 2.7 JOIN的索引策略

**为什么重要**：JOIN是性能瓶颈，索引能大幅提升速度。

- 在JOIN键（ON条件中的列）上建立索引
- 小表驱动大表（把行数少的表放在JOIN前面）
- EXPLAIN看type字段：ALL(最差) → index → range → ref → eq_ref → const(最好)

---

## 三、动手练习

In [ ]:
import sqlite3
import pandas as pd
import numpy as np

np.random.seed(42)
conn = sqlite3.connect(':memory:')

# 用户表（5个用户）
pd.DataFrame({
    'user_id': [1, 2, 3, 4, 5],
    'name': ['Alice', 'Bob', 'Charlie', 'David', 'Eve'],
    'city': ['北京', '上海', '北京', '广州', '深圳']
}).to_sql('users', conn, index=False)

# 订单表（user_id=2有2单，user_id=5没有订单）
pd.DataFrame({
    'order_id': [101, 102, 103, 104, 105],
    'user_id': [1, 2, 2, 3, 4],
    'amount': [100, 200, 150, 300, 250]
}).to_sql('orders', conn, index=False)

# 退款表（含NULL陷阱）
pd.DataFrame({
    'order_id': [101, None],
    'refund_amount': [100, 50]
}).to_sql('refunds', conn, index=False)

print("users表: 5行")
print("orders表: 5行")
print("refunds表: 2行（含1个NULL）")
print()

### 练习1：INNER JOIN vs LEFT JOIN——行数和内容对比

In [ ]:
print("=" * 60)
print("练习1A：INNER JOIN——只保留有订单的用户")
q1a = """SELECT u.user_id, u.name, COUNT(o.order_id) AS order_cnt,
    COALESCE(SUM(o.amount), 0) AS total_amount
FROM users u
INNER JOIN orders o ON u.user_id = o.user_id
GROUP BY u.user_id, u.name"""
print(pd.read_sql(q1a, conn))
print("  注意：user_id=5 (Eve) 消失了——她没有订单")

print()
print("=" * 60)
print("练习1B：LEFT JOIN——保留所有用户")
q1b = """SELECT u.user_id, u.name, COUNT(o.order_id) AS order_cnt,
    COALESCE(SUM(o.amount), 0) AS total_amount
FROM users u
LEFT JOIN orders o ON u.user_id = o.user_id
GROUP BY u.user_id, u.name"""
print(pd.read_sql(q1b, conn))
print("  注意：user_id=5 (Eve) 保留了——order_cnt=0, total_amount=0")
print()
print("关键差异：'有多少用户没下过单'——LEFT JOIN才能回答这个问题")

### 练习2：LEFT JOIN + WHERE陷阱验证

In [ ]:
print("=" * 60)
print("练习2：LEFT JOIN + WHERE右表条件 = INNER JOIN（陷阱验证）")

print()
print("写法A：条件在ON里（正确，保留所有用户）")
q2a = """SELECT u.user_id, u.name, o.amount
FROM users u
LEFT JOIN orders o ON u.user_id = o.user_id AND o.amount > 150"""
r2a = pd.read_sql(q2a, conn)
print(r2a)
print(f"  行数: {len(r2a)}（= users表的5行，Eve的amount为NaN）")

print()
print("写法B：条件在WHERE里（错误！变成INNER JOIN）")
q2b = """SELECT u.user_id, u.name, o.amount
FROM users u
LEFT JOIN orders o ON u.user_id = o.user_id
WHERE o.amount > 150"""
r2b = pd.read_sql(q2b, conn)
print(r2b)
print(f"  行数: {len(r2b)}（只有满足amount>150的，Eve丢了！）")

print()
print("结论：WHERE o.amount > 150 会过滤掉 amount IS NULL 的行")
print("= 把LEFT JOIN变成了INNER JOIN！")

### 练习3：一对多JOIN行数膨胀验证

In [ ]:
print("=" * 60)
print("练习3：一对多JOIN行数膨胀")

print(f"users表行数: {pd.read_sql('SELECT COUNT(*) FROM users', conn).iloc[0,0]}")
print(f"orders表行数: {pd.read_sql('SELECT COUNT(*) FROM orders', conn).iloc[0,0]}")

q3 = "SELECT COUNT(*) AS cnt FROM users u LEFT JOIN orders o ON u.user_id = o.user_id"
joined_rows = pd.read_sql(q3, conn).iloc[0,0]
print(f"JOIN后行数: {joined_rows}")
print(f"膨胀: +{joined_rows - 5}行（因为user_id=2有2个订单）")

print()
print("如果JOIN后对orders.amount做SUM——user_id=2的amount会被重复加2次！")
print("正确做法：先聚合orders表，再JOIN users表")

### 练习4：NOT IN陷阱 + NOT EXISTS对比

In [ ]:
print("=" * 60)
print("练习4：NOT IN陷阱验证")

print()
print("refunds表内容（含NULL）：")
print(pd.read_sql("SELECT * FROM refunds", conn))

print()
print("1) NOT IN —— 返回空！")
q4a = "SELECT * FROM orders WHERE order_id NOT IN (SELECT order_id FROM refunds)"
r4a = pd.read_sql(q4a, conn)
print(f"  结果行数: {len(r4a)}（应该是4行，但NOT IN返回了空集！）")

print()
print("2) NOT EXISTS —— 正确返回4行")
q4b = """SELECT * FROM orders o
WHERE NOT EXISTS (SELECT 1 FROM refunds r WHERE r.order_id = o.order_id)"""
r4b = pd.read_sql(q4b, conn)
print(f"  结果行数: {len(r4b)}")
print(r4b[['order_id', 'user_id', 'amount']])

print()
print("3) LEFT JOIN + IS NULL —— 正确返回4行")
q4c = """SELECT o.* FROM orders o
LEFT JOIN refunds r ON o.order_id = r.order_id
WHERE r.order_id IS NULL"""
r4c = pd.read_sql(q4c, conn)
print(f"  结果行数: {len(r4c)}")

print()
print("结论：永远不要用NOT IN——改用NOT EXISTS或LEFT JOIN + IS NULL")

### 练习5：自连接——找出同城市消费更高的其他用户

In [ ]:
print("=" * 60)
print("练习5：自连接——同城市内，消费金额比较")

q5 = """WITH user_total AS (
    SELECT u.user_id, u.name, u.city,
        COALESCE(SUM(o.amount), 0) AS total_amount
    FROM users u
    LEFT JOIN orders o ON u.user_id = o.user_id
    GROUP BY u.user_id, u.name, u.city
)
SELECT a.name AS user1, a.total_amount AS amount1,
       b.name AS user2, b.total_amount AS amount2,
       a.city
FROM user_total a
JOIN user_total b ON a.city = b.city AND a.user_id < b.user_id
ORDER BY a.city, a.total_amount DESC"""
print(pd.read_sql(q5, conn))
print()
print("自连接使用场景：
  1. 行间比较（同一城市内用户互相比较）
  2. 层级数据（员工-经理关系）
  3. 连续行为（前后两天登录）")

---

## 四、业务场景应用

### 场景1：统计已下单但未支付的用户

运营需要催付——找出下单后24小时内未支付的用户。

```sql
SELECT o.user_id, o.order_id, o.order_time
FROM orders o
LEFT JOIN payments p ON o.order_id = p.order_id
WHERE p.order_id IS NULL
  AND o.order_time >= DATE('now', '-1 day');
```

**为什么用LEFT JOIN + IS NULL而不是NOT IN**：因为payments表可能order_id为NULL（退款单等），NOT IN会有陷阱。

### 场景2：用户表和订单表LEFT JOIN后行数暴涨

用户表10万行，LEFT JOIN订单表后变成50万行。

**排查**：
1. 确认是一对多关系——一个用户多个订单
2. 如果是要统计用户级别的指标，先聚合订单表再JOIN
3. 如果确实需要保留明细，统计用户数时用COUNT(DISTINCT user_id)

### 场景3：多表JOIN的过滤条件策略

订单表 JOIN 用户表 JOIN 商品表，每个表都有过滤条件：
- 用户表"只看北京"→ ON users中（保护LEFT JOIN语义）
- 商品表"只看数码"→ ON products中
- 订单表"只看1月"→ WHERE中（左表过滤）

### 场景4：全量表与增量表的合并

使用FULL OUTER JOIN合并全量表(昨天的全量)和增量表(今天的新增和变更)：
```sql
SELECT COALESCE(f.user_id, i.user_id) AS user_id,
       COALESCE(i.name, f.name) AS name,
       COALESCE(i.amount, f.amount) AS amount
FROM full_table f
FULL OUTER JOIN incremental_table i ON f.user_id = i.user_id;
```

---

## 五、分级检验清单

### 初级检验（必须能独立完成）
- [ ] 能画出四种JOIN的韦恩图
- [ ] 能一句话说清ON和WHERE的区别
- [ ] 能写出LEFT JOIN + IS NULL找出"有但没有"的模式
- [ ] 能区分INNER JOIN和LEFT JOIN的适用场景

**答案提示**：
- ON在JOIN时执行，WHERE在JOIN后执行
- LEFT JOIN + IS NULL = 左表有但右表没有
- INNER JOIN只保留交集，LEFT JOIN保留左表全部

### 中级检验（能解释给同事听）
- [ ] 能构造数据验证LEFT JOIN + WHERE右表条件 = INNER JOIN
- [ ] 能解释为什么NOT IN子查询中有NULL时返回空
- [ ] 能分析一对多JOIN后的行数膨胀是否合理
- [ ] 能写出自连接的SQL

**答案提示**：
- WHERE会过滤NULL行，所以LEFT JOIN变成INNER JOIN
- NOT IN要求所有值都不等于，NULL参与比较返回UNKNOWN
- 一对多膨胀是正常的，但统计时需要先聚合或去重

### 高级检验（能举一反三）
- [ ] 能设计多表JOIN的索引策略
- [ ] 能用EXPLAIN分析JOIN性能瓶颈
- [ ] 能处理复杂的层级数据（如组织架构递归）
- [ ] 能写出高效的复购率统计SQL（避免多次JOIN）

**答案提示**：
- 在JOIN键上建索引，小表驱动大表
- EXPLAIN看type和rows字段
- 层级数据用递归CTE
- 复购率用CASE WHEN聚合，避免自连接

---

## 六、跨模块连接

| 方向 | 关联内容 | 说明 |
|------|---------|------|
| **← 基础** | SELECT/GROUP BY | JOIN是FROM的扩展 |
| **← 本周二** | 窗口函数 | 很多窗口函数的需求可以用JOIN实现（但窗口函数更优雅） |
| **→ 昨日** | 留存计算 | 留存SQL需要自连接或LEFT JOIN |
| **→ 第7周** | 数仓星型模型 | 事实表需要JOIN多个维度表 |
| **业务连接** | 所有多表分析 | 实际业务分析很少只用单表 |
| **扩展阅读** | 02analyst_thinking W1D3 | JOIN完整知识图谱+面试题 |

### 横向连接：JOIN与Pandas的对等操作

| SQL JOIN | Pandas对等 |
|---------|-----------|
| INNER JOIN | pd.merge(left, right, on='key', how='inner') |
| LEFT JOIN | pd.merge(left, right, on='key', how='left') |
| RIGHT JOIN | pd.merge(left, right, on='key', how='right') |
| FULL JOIN | pd.merge(left, right, on='key', how='outer') |
| CROSS JOIN | pd.merge(left.assign(key=1), right.assign(key=1), on='key').drop('key', 1) |
| SEMI JOIN | left[left['key'].isin(right['key'])] |
| ANTI JOIN | left[~left['key'].isin(right['key'])] |

---

## 七、当日小结

> **今天最重要的一个认知**：JOIN之前先确认"输出的一行代表什么"，JOIN之后先检查行数是否合理。行数膨胀是最隐蔽的JOIN bug——它不会报错，只会让你的SUM变成实际值的N倍。LEFT JOIN + WHERE右表条件 = INNER JOIN——这行代码值得贴在显示器上。NOT IN有NULL陷阱，永远用NOT EXISTS或LEFT JOIN + IS NULL替代。

**三条金句带回家**：
1. LEFT JOIN + WHERE右表条件 = INNER JOIN——最容易踩的坑，没有之一
2. NOT IN在子查询有NULL时返回空——改用NOT EXISTS或LEFT JOIN + IS NULL
3. 一对多JOIN会导致行数膨胀——先聚合再JOIN，或在统计时用COUNT(DISTINCT)

### 课后拓展

1. **练习**：用JOIN实现学生成绩排名（自连接比较）

2. **思考**：如果订单表和支付表都是10亿行，JOIN会有什么问题？如何优化？

3. **预习**：明天要学的CTE与子查询可以让复杂的JOIN查询更可读

4. **面试准备**：背熟以下问题——"INNER JOIN和LEFT JOIN的区别？""为什么LEFT JOIN + WHERE会变成INNER JOIN？""NOT IN和NOT EXISTS的区别？"

---

*03analyst_thinking · Week 5 Day 4 · 2026-05-21*